# Phase 2e: Choosing a working Hugging Face CLAP checkpoint (training tracks only)

Phase 2d found that the HF `laion/larger_clap_music` conversion behaves as if broken (unrelated text prompts at cosine
0.999, zero-shot mood and genre at chance; LAION-AI/CLAP issue #126). This notebook records, reproducibly, how the
replacement was chosen. Three HF CLAP checkpoints are compared with checks that need no retrieval results:

* `logit_scale_a` (a trained CLAP has a large learned temperature; ≈ 1 means untrained or lost),
* mean cosine between the text embeddings of unrelated prompts (should be well below 1),
* median cosine between the audio embeddings of different tracks (a collapsed space is near 1),
* zero-shot genre accuracy (7 AllMusic top-level genres, prompts "<genre> music"; evaluation metadata only),
* zero-shot quadrant accuracy with the Phase 2d mood prompts.

**Only tracks from the shared 70/15/15 training partition are used** (20 per genre for tracks with exactly one of the
seven genres; 50 per quadrant; seed 42), and only the first 10-second window of each excerpt, so the choice uses no
validation or test track. Output: `clap_checkpoint_check.csv` (the earlier ad-hoc `clap_checkpoint_validation.csv`
is kept for reference).

In [1]:
import os
os.environ.setdefault('HF_HUB_OFFLINE', '1')
import numpy as np
import pandas as pd
import torch
import librosa
from transformers import ClapModel, ClapProcessor

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR = os.path.join(ROOT, 'outputs')
SEED, SR = 42, 48_000
QUADS = ['Q1', 'Q2', 'Q3', 'Q4']
GENRES = ['Pop/Rock', 'Electronic', 'Country', 'Rap', 'Blues', 'Jazz', 'Reggae']
CHECKPOINTS = ['laion/larger_clap_music', 'laion/clap-htsat-unfused', 'laion/larger_clap_general']
torch.set_num_threads(os.cpu_count())

merged = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))
mm = merged[merged['is_multimodal']].set_index('song_id')
lyr_stem_to_sid = {os.path.splitext(os.path.basename(p))[0]: s for s, p in zip(mm.index, mm['lyrics_path'])}
sc = 'tvt_70_15_15'
a_tr = set(pd.read_csv(os.path.join(ROOT, 'MERGE_Audio_Balanced', 'tvt_dataframes', sc, f'{sc}_train_audio_balanced.csv'))['Song'])
l_tr = set(pd.read_csv(os.path.join(ROOT, 'MERGE_Lyrics_Balanced', 'tvt_dataframes', sc, f'{sc}_train_lyrics_balanced.csv'))['Song']
           .map(lyr_stem_to_sid).dropna())
train = mm.loc[sorted(a_tr & l_tr & set(mm.index))]
print('shared 70/15/15 training tracks:', len(train))

top = train['genres'].astype(str).str.split(',').apply(lambda g: [x.strip() for x in g if x.strip() in GENRES])
single = train[top.str.len() == 1].assign(genre=top[top.str.len() == 1].str[0])
genre_ids = single.groupby('genre', group_keys=False).apply(lambda g: g.sample(min(20, len(g)), random_state=SEED))
mood_ids = train.groupby('quadrant', group_keys=False).apply(lambda g: g.sample(50, random_state=SEED))
print('genre sample:', genre_ids.genre.value_counts().to_dict(), '| quadrant sample:', mood_ids.quadrant.value_counts().to_dict())
assert not (set(genre_ids.index) | set(mood_ids.index)) - set(train.index)

shared 70/15/15 training tracks: 1187
genre sample: {'Blues': 20, 'Country': 20, 'Electronic': 20, 'Jazz': 20, 'Pop/Rock': 20, 'Rap': 20, 'Reggae': 19} | quadrant sample: {'Q1': 50, 'Q2': 50, 'Q3': 50, 'Q4': 50}


<HOME>\AppData\Local\Temp\ipykernel_92800\2164284093.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  genre_ids = single.groupby('genre', group_keys=False).apply(lambda g: g.sample(min(20, len(g)), random_state=SEED))
<HOME>\AppData\Local\Temp\ipykernel_92800\2164284093.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mood_ids = train.groupby('quadrant', group_keys=False).apply(lambda g: g.sample(50

In [2]:
PROMPTS = {
    'Q1': ['happy music', 'upbeat, joyful music', 'cheerful and energetic music', 'exciting, euphoric music'],
    'Q2': ['angry music', 'aggressive, tense music', 'furious and intense music', 'anxious, menacing music'],
    'Q3': ['sad music', 'melancholic, depressing music', 'sorrowful and gloomy music', 'lonely, heartbroken music'],
    'Q4': ['calm music', 'relaxing, peaceful music', 'serene and soothing music', 'tender, content music'],
}
UNRELATED = ['a dog barking', 'heavy metal guitar solo', 'a quiet string quartet', 'rain on a window',
             'a man giving a speech', 'electronic dance music with a strong beat']
l2 = lambda X: X / np.linalg.norm(X, axis=-1, keepdims=True)

ids_all = sorted(set(genre_ids.index) | set(mood_ids.index))
wave = {s: librosa.load(mm.loc[s, 'audio_path'], sr=SR, mono=True, duration=10)[0] for s in ids_all}

rows = []
for ck in CHECKPOINTS:
    model = ClapModel.from_pretrained(ck).eval(); proc = ClapProcessor.from_pretrained(ck)
    def text(t):
        with torch.no_grad():
            return l2(model.get_text_features(**proc(text=t, return_tensors='pt', padding=True)).numpy())
    A = {}
    for i in range(0, len(ids_all), 8):
        b = ids_all[i:i + 8]
        with torch.no_grad():
            E = model.get_audio_features(**proc(audio=[wave[s] for s in b], sampling_rate=SR, return_tensors='pt')).numpy()
        A.update(zip(b, l2(E)))
    T = text(UNRELATED); tc = (T @ T.T)[np.triu_indices(len(T), 1)].mean()
    Aall = np.vstack([A[s] for s in ids_all]); ac = np.median((Aall @ Aall.T)[np.triu_indices(len(Aall), 1)])
    G = text([f'{g} music' for g in GENRES])
    gp = np.array(GENRES)[(np.vstack([A[s] for s in genre_ids.index]) @ G.T).argmax(1)]
    Q = l2(np.vstack([text(PROMPTS[q]).mean(0) for q in QUADS]))
    qp = np.array(QUADS)[(np.vstack([A[s] for s in mood_ids.index]) @ Q.T).argmax(1)]
    rows.append({'checkpoint': ck, 'commit': getattr(model.config, '_commit_hash', None),
                 'logit_scale_a': float(model.logit_scale_a.exp()), 'text_unrelated_cos_mean': tc,
                 'audio_pair_cos_median': ac, 'genre_zero_shot_acc': np.mean(gp == genre_ids.genre.values),
                 'genre_chance': 1 / len(GENRES), 'genre_predicted_classes': len(set(gp)),
                 'mood_zero_shot_acc': np.mean(qp == mood_ids.quadrant.values),
                 **{f'mood_pred_share_{q}': np.mean(qp == q) for q in QUADS}})
    print(rows[-1], flush=True)
    del model
res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, 'clap_checkpoint_check.csv'), index=False)
pd.set_option('display.width', 250)
print(res.round(3).to_string(index=False))

{'checkpoint': 'laion/larger_clap_music', 'commit': 'a0b4534a14f58e20944452dff00a22a06ce629d1', 'logit_scale_a': 1.027732253074646, 'text_unrelated_cos_mean': 0.99917054, 'audio_pair_cos_median': 0.97171545, 'genre_zero_shot_acc': 0.1366906474820144, 'genre_chance': 0.14285714285714285, 'genre_predicted_classes': 2, 'mood_zero_shot_acc': 0.2, 'mood_pred_share_Q1': 0.575, 'mood_pred_share_Q2': 0.0, 'mood_pred_share_Q3': 0.0, 'mood_pred_share_Q4': 0.425}


<HOME>\AppData\Local\Temp\ipykernel_92800\3614512485.py:33: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  'logit_scale_a': float(model.logit_scale_a.exp()), 'text_unrelated_cos_mean': tc,


{'checkpoint': 'laion/clap-htsat-unfused', 'commit': '8fa0f1c6d0433df6e97c127f64b2a1d6c0dcda8a', 'logit_scale_a': 18.661176681518555, 'text_unrelated_cos_mean': 0.16942237, 'audio_pair_cos_median': 0.7125101, 'genre_zero_shot_acc': 0.15827338129496402, 'genre_chance': 0.14285714285714285, 'genre_predicted_classes': 5, 'mood_zero_shot_acc': 0.445, 'mood_pred_share_Q1': 0.755, 'mood_pred_share_Q2': 0.19, 'mood_pred_share_Q3': 0.01, 'mood_pred_share_Q4': 0.045}


{'checkpoint': 'laion/larger_clap_general', 'commit': 'ada0c23a36c4e8582805bb38fec3905903f18b41', 'logit_scale_a': 38.66471862792969, 'text_unrelated_cos_mean': 0.25408986, 'audio_pair_cos_median': 0.6141714, 'genre_zero_shot_acc': 0.5251798561151079, 'genre_chance': 0.14285714285714285, 'genre_predicted_classes': 7, 'mood_zero_shot_acc': 0.54, 'mood_pred_share_Q1': 0.405, 'mood_pred_share_Q2': 0.39, 'mood_pred_share_Q3': 0.19, 'mood_pred_share_Q4': 0.015}


               checkpoint                                   commit  logit_scale_a  text_unrelated_cos_mean  audio_pair_cos_median  genre_zero_shot_acc  genre_chance  genre_predicted_classes  mood_zero_shot_acc  mood_pred_share_Q1  mood_pred_share_Q2  mood_pred_share_Q3  mood_pred_share_Q4
  laion/larger_clap_music a0b4534a14f58e20944452dff00a22a06ce629d1          1.028                    0.999                  0.972                0.137         0.143                        2               0.200               0.575                0.00                0.00               0.425
 laion/clap-htsat-unfused 8fa0f1c6d0433df6e97c127f64b2a1d6c0dcda8a         18.661                    0.169                  0.713                0.158         0.143                        5               0.445               0.755                0.19                0.01               0.045
laion/larger_clap_general ada0c23a36c4e8582805bb38fec3905903f18b41         38.665                    0.254                  0.614 

Selection rule (fixed before looking at retrieval): discard a checkpoint whose text space is collapsed
(unrelated prompts at cosine > 0.9) or whose zero-shot genre accuracy is not clearly above chance; among the rest, take
the highest zero-shot genre accuracy. Genre is used rather than mood because it is not an evaluation target of this
study.